# Resilience: what breaks when the border router, the internet or the vendor goes away

**Claim cluster:** `C019`, `C020`, `C021`, `C022`, `C026`, `C027`, `C070`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "smarthome.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/smarthome.duckdb (read-only).
CLAIM_IDS = ('C019', 'C020', 'C021', 'C022', 'C026', 'C027', 'C070')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

This cluster is almost entirely prose evidence: the corpus records one quantified number about losing device control (`M006`, `about 1` minute) and describes the rest in claims. The provenance panels are therefore the substance of this page, not an appendix to it.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Three outage states, four capabilities, one claim per cell

The corpus distinguishes local infrastructure failure (the Thread border router), network failure (the WAN link) and lifecycle failure (the vendor retiring a cloud service). Those are three different failure classes with different consequences, and the corpus's own recommendation claim `C076` argues manufacturers should publish failures in exactly this classified way. The map below applies that structure to the claims that exist: it is a reading of the claims, and an unrecorded cell stays blank.

In [ ]:
# Reading scale, as in the ecosystem notebook:
#   1   the claim states the capability keeps working in that state
#   0   the claim states it fails, is lost, or is blocked
#   no entry = no claim in this cluster records that combination
STATES = ["Border router removed / replaced", "Internet (WAN) down",
          "Vendor cloud retired"]
CAPABILITIES = ["Basic control of commissioned devices",
                "Commissioning new devices", "Firmware updates",
                "Local Matter/Thread paths"]
FAILURE_CELLS = {
    ("Basic control of commissioned devices",
     "Border router removed / replaced"): ("C020", 0),
    ("Commissioning new devices",
     "Border router removed / replaced"): ("C021", 0),
    ("Firmware updates", "Border router removed / replaced"): ("C019", 0),
    ("Local Matter/Thread paths",
     "Border router removed / replaced"): ("C026", 0),
    ("Basic control of commissioned devices", "Internet (WAN) down"): ("C027", 1),
    ("Local Matter/Thread paths", "Internet (WAN) down"): ("C027", 1),
    ("Basic control of commissioned devices", "Vendor cloud retired"): ("C070", 0),
}

_claims_by_id = claims.set_index("local_id")
_rows, _texts, _hovers = [], [], []
for _capability in CAPABILITIES:
    _row, _row_text, _row_hover = [], [], []
    for _state in STATES:
        _cid, _status = FAILURE_CELLS.get((_capability, _state), (None, None))
        _row.append(_status)
        _row_text.append(_cid or "")
        if _cid is None:
            _row_hover.append(f"{_state}<br>{_capability}"
                              "<br>no claim in this cluster records this cell")
            continue
        _claim = _claims_by_id.loc[_cid]
        _row_hover.append(
            f"{_state}<br>{_capability}<br><b>{_cid}</b>"
            f" [{_claim['claim_type']}, {_claim['confidence']} confidence]"
            f"<br>{_claim['claim_text']}"
        )
    _rows.append(_row)
    _texts.append(_row_text)
    _hovers.append(_row_hover)

fig = go.Figure(go.Heatmap(
    z=_rows,
    x=STATES,
    y=CAPABILITIES,
    zmin=0,
    zmax=1,
    colorscale=[[0.0, "#e8b6bb"], [0.5, "#f0f0f0"], [1.0, "#bcd9c4"]],
    xgap=3,
    ygap=3,
    text=_texts,
    texttemplate="%{text}",
    textfont=dict(size=12),
    customdata=_hovers,
    hovertemplate="%{customdata}<extra></extra>",
    colorbar=dict(title="claim reading", tickmode="array", tickvals=[0, 1],
                  ticktext=["fails / blocked", "keeps working"]),
))
fig.update_layout(
    title=("Failure map by outage state, one claim per cell "
           "(blank = no claim in this cluster)"),
    height=420,
    xaxis=dict(side="top"),
    yaxis=dict(autorange="reversed"),
)
fig.show()

# The same cells, counted: how much of the map the corpus actually filled.
_coverage = []
for _state in STATES:
    _recorded = [FAILURE_CELLS[(_cap, _state)]
                 for _cap in CAPABILITIES if (_cap, _state) in FAILURE_CELLS]
    _fails = sum(1 for _cid, _status in _recorded if _status == 0)
    _coverage.append({
        "state": _state,
        "capabilities_in_map": len(CAPABILITIES),
        "recorded_as_failing": _fails,
        "recorded_as_working": len(_recorded) - _fails,
        "no_claim_recorded": len(CAPABILITIES) - len(_recorded),
    })
coverage = pd.DataFrame(_coverage)
show(coverage)
fig = go.Figure()
fig.add_trace(go.Bar(name="recorded as failing", x=coverage["state"],
                     y=coverage["recorded_as_failing"], marker_color="#b32637"))
fig.add_trace(go.Bar(name="recorded as still working", x=coverage["state"],
                     y=coverage["recorded_as_working"], marker_color="#1a7f37"))
fig.add_trace(go.Bar(name="no claim recorded", x=coverage["state"],
                     y=coverage["no_claim_recorded"], marker_color="#d9d9d9"))
fig.update_layout(
    barmode="stack",
    title=("How much of the failure map the corpus fills "
           "(four capabilities per state)"),
    yaxis_title="capabilities",
    height=340,
)
fig.show()

## Supporting context

### The one quantity, and two figures that are not comparable with it

`M006` is the only recorded measurement of how quickly control is lost when the border router disappears. The corpus also records a protocol-maturity figure (`M007`, Z-Wave's about 20 years) and an onboarding response expectation from its own test procedure (`M088`, 2 seconds). They are drawn together to show the *scale* of the response-time language the corpus uses, each on its own unit: they are not three points on one measurement axis.

In [ ]:
display(HTML(
    "<h4>Recorded time figures in the resilience cluster</h4>"
    "<p><code>M006</code> is a measured consequence; <code>M007</code> is a "
    "protocol-maturity statement; <code>M088</code> is the corpus's own test "
    "expectation. The units differ, so each bar is labelled with its own.</p>"
))
_timed = viz.metrics_by_id(["M006", "M007", "M088"])
show(_timed[["local_id", "metric_name", "value", "unit", "confidence",
             "source_ref", "section"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))

fig = go.Figure(go.Bar(
    x=[viz.numeric_value(value) for value in _timed["value"]],
    y=[f"{record['local_id']} {record['metric_name']} ({record['unit']})"
       for record in _timed.to_dict("records")],
    orientation="h",
    marker=dict(color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                       for c in _timed["confidence"]]),
    customdata=list(zip(_timed["value"], _timed["unit"], _timed["confidence"])),
    hovertemplate=("%{y}<br>stated: %{customdata[0]} %{customdata[1]}"
                   "<br>confidence: %{customdata[2]}<extra></extra>"),
))
fig.update_layout(
    title=("Three recorded time figures, each in its own unit "
           "(one axis, three clocks: minutes, years, seconds)"),
    xaxis_title="value, in the unit named on the bar",
    height=300,
    yaxis=dict(automargin=True),
)
fig.show()
display(HTML(
    "<h4>Claim cluster evidence table</h4>"
    "<p>Every claim in this cluster, with its evidence class, confidence, "
    "workflow stage and cited source ids. This is the table the figures above "
    "are built from.</p>"
))
show(
    claims[["local_id", "claim_type", "evidence_marker", "confidence",
            "workflow_stage", "source_refs", "falsifier_missing", "section"]]
    .rename(columns={
        "local_id": "claim", "claim_type": "type", "evidence_marker": "ev",
        "confidence": "conf", "workflow_stage": "stage",
        "source_refs": "sources", "falsifier_missing": "no_falsifier",
    })
)

display(HTML("<h4>Sources cited by the cluster</h4>"))
show(sources[["local_id", "title", "publisher", "classification",
              "publication_date", "url"]].rename(
    columns={"local_id": "source"}))

## Uncertainty and gaps

### Two claims that look contradictory and are not

`C020` states that Matter-over-Thread devices lose the ability to communicate with each other when the border router is down. `C027` states that local Matter/Thread paths work without internet for basic operations. Both are in the map, and they are not in conflict: `C027` is about an internet outage on a working mesh, `C020` is about the mesh losing its route to the IP network. Merging them into one 'does local control work?' verdict would destroy that distinction, so the map keeps the two outage states separate.

Other limits on this page:

- **One measurement, several relayed observations.** `M006` is medium confidence; the cell readings come from `reported signal` claims (`C019`, `C020`, `C021`, `C027`) rather than from controlled trials.
- **`C022` is a low-confidence inference** that Z-Wave remains the most reliable option for mission-critical applications. It is kept because it is the corpus's only comparative reliability statement, and it is labelled as an inference wherever it appears.
- **`C070` names vendors without dates.** Neato, Wemo and Nest are recorded as having ended cloud support earlier than promised, but the corpus records no shutdown dates, so no timeline is drawn.
- **The vendor-lifecycle column is one cell deep.** Only basic control is claimed against a retired cloud; commissioning, firmware and local-path behaviour after a cloud shutdown are simply not recorded.

In [ ]:
# The 'still works / fails' distinction is re-checked against the claim text
# here rather than trusted from the mapping: the table prints the wording the
# reading was made from.
_cell_claims = sorted({cid for cid, _status in FAILURE_CELLS.values()})
display(HTML(
    "<h4>The claims the map is read from</h4>"
    "<p>Each reading in the failure map comes from one of these claims. The "
    "quotations are the claim text as stored in the database.</p>"
))
show(pd.DataFrame([
    {"claim": cid, "type": claims.set_index("local_id").loc[cid, "claim_type"],
     "confidence": claims.set_index("local_id").loc[cid, "confidence"],
     "no_falsifier": claims.set_index("local_id").loc[cid, "falsifier_missing"],
     "claim_text": claims.set_index("local_id").loc[cid, "claim_text"]}
    for cid in _cell_claims
]))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

All claim text comes from `viz.fetch_claims`; the three time figures from `viz.metrics_by_id`, parsed with `viz.numeric_value`. The map's cell readings are notebook-level data and are printed with the claim text they were read from, so the mapping can be audited without leaving the page.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()